# Eff-3DPSeg on Kaggle (downloads Pheno4D directly)

Settings (right panel): **Accelerator = GPU** (prefer **T4**), **Internet = On** (needed to clone the repo and download the 4.4 GB data).

To keep it running without the browser: **Save Version -> Save & Run All (Commit)**. Checkpoints and predictions in `/kaggle/working` are saved with the version. A session is limited to about 12 hours in total, including the download and preparation.

Raw data and converted `.npz` files go to scratch space (`/kaggle/temp`), which is not saved with the version.

In [ ]:
import os, subprocess, torch
REPO_URL = 'https://github.com/gitmate21/3d-phenomics.git'
REPO = '/kaggle/working/3d-phenomics'

print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Notebook Settings -> Accelerator -> GPU, then Run -> Restart session.')
print(torch.cuda.get_device_name(0), '| compute capability', torch.cuda.get_device_capability(0))

if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO], check=True)
%cd {REPO}/eff3dpseg

# spconv has no working build for CUDA 13 (its C++ build fails): use the pure-PyTorch backend.
# Later `!python scripts/...` cells inherit this environment variable.
os.environ['EFF_SPARSE_BACKEND'] = 'torch'
CKPT_DIR, LOG_DIR, PRED_DIR = '/kaggle/working/checkpoints', '/kaggle/working/logs', '/kaggle/working/preds'
for d in (CKPT_DIR, LOG_DIR, PRED_DIR):
    os.makedirs(d, exist_ok=True)

In [ ]:
!python scripts/test_sparse_torch.py
!python scripts/smoke_test.py

In [ ]:
# Download Pheno4D directly from the University of Bonn (4.4 GB zip; several minutes).
# Check the dataset's terms on https://www.ipb.uni-bonn.de/data/pheno4d/ before using or sharing it.
PHENO4D_URL = 'https://www.ipb.uni-bonn.de/html/projects/Pheno4D/Pheno4D.zip'
SCRATCH = '/kaggle/temp' if os.path.isdir('/kaggle/temp') else '/tmp'
RAW_ROOT = f'{SCRATCH}/pheno4d_raw'
!df -h {SCRATCH} | tail -1
if not os.path.isdir(RAW_ROOT):
    !wget -nv -O {SCRATCH}/Pheno4D.zip {PHENO4D_URL}
    !unzip -q {SCRATCH}/Pheno4D.zip -d {RAW_ROOT}
    !rm {SCRATCH}/Pheno4D.zip
!ls {RAW_ROOT}

In [ ]:
# RAW_DIR must be the folder that directly contains the plant folders (Maize01, Tomato01, ...).
# The zip's internal layout is not assumed: find where Maize01 lives.
import glob
hits = glob.glob(f'{RAW_ROOT}/**/Maize01', recursive=True)
assert hits, f'Maize01 not found under {RAW_ROOT}; look at the listing above'
RAW_DIR = os.path.dirname(hits[0])
print('RAW_DIR =', RAW_DIR)
!ls {RAW_DIR}

In [ ]:
# Convert raw txt -> npz + plant-level splits (whole plants are held out, so no leakage).
# Reading ~50 MB text files is slow (several minutes for the whole set); this runs once.
SPECIES = 'Tomato'                        # 'Tomato' (Tomato01-07) or 'Maize' (Maize01-07); one species per run
DATA = f'{SCRATCH}/pheno4d_npz_{SPECIES.lower()}'
plants = sorted(d for d in os.listdir(RAW_DIR) if os.path.isdir(f'{RAW_DIR}/{d}') and d.startswith(SPECIES))
print('plants:', plants)
VAL, TEST = plants[-2], plants[-1]        # edit to hold out other plants
print('val:', VAL, '| test:', TEST)
!head -n 3 {RAW_DIR}/{plants[0]}/$(ls {RAW_DIR}/{plants[0]} | grep _a.txt | head -1)
!python scripts/prepare_pheno4d.py --raw {RAW_DIR} --out {DATA} --plants {' '.join(plants)} --val-plants {VAL} --test-plants {TEST}

### Optional: continue earlier runs
If you saved checkpoints (for example `tomato_w50_latest.pth`) from a previous session, add them as a Kaggle Dataset, attach it to this notebook, and copy them to `CKPT_DIR` before training. Training then continues with `--resume auto`:

`!cp /kaggle/input/<your-checkpoint-dataset>/*.pth {CKPT_DIR}/`

Each run writes `<run>_params.json` (all settings and directories) to `LOG_DIR` and `CKPT_DIR`; keep it together with the checkpoint.

In [ ]:
# ---------------- run settings (printed and saved with every run as <run>_params.json) ----------------
VOXEL_SIZE = 0.2                  # <-- SET from the printed extent (plant ~50-200 voxels tall; the paper used 0.5 for tomato)
LABELED_POINTS = [50, 100, 200]   # one training run per value (the paper's settings). 0 = all points labeled
DROP_SOIL = True                  # True: delete soil points as the paper does. False: keep them as input
MAX_ITER, BATCH_SIZE = 2000, 1    # the paper used 4000 iterations with batch size 2
HOURS_PER_RUN = 3.3               # each run saves and stops at this limit; --resume auto continues it later
# Quick trial first: MAX_ITER = 20 and LABELED_POINTS = [50] shows the run report, speed and memory in a few minutes.
# --------------------------------------------------------------------------------------------------------
SOIL = '--drop-soil' if DROP_SOIL else '--keep-soil'
RUNS = [f"{SPECIES.lower()}_{'full' if k == 0 else f'w{k}'}" for k in LABELED_POINTS]
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
for k, run in zip(LABELED_POINTS, RUNS):
    print(f"\n################ {run}: {k or 'all'} labeled points ################")
    !python scripts/train.py --data {DATA} --run-name {run} --labeled-points {k} {SOIL} \
        --max-iter {MAX_ITER} --batch-size {BATCH_SIZE} --voxel-size {VOXEL_SIZE} --validate-step 50 --save-every 50 \
        --ckpt-dir {CKPT_DIR} --log-dir {LOG_DIR} --resume auto --time-limit-hours {HOURS_PER_RUN}

In [ ]:
# Evaluate every run's best checkpoint on the validation and test plants (all raw points, soil excluded) and print
# precision / recall / F1 / IoU next to the matching paper rows (Table 3, tomato). Voxel size and soil handling are
# read from each checkpoint. Use --which latest to evaluate the final iteration instead of the best one.
!python scripts/compare_runs.py --data {DATA} --ckpt-dir {CKPT_DIR} --runs {' '.join(RUNS)} --phases val test